In [163]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torchmetrics import Accuracy, F1Score
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
df = pd.read_csv('data/joined_with_tt_scores.csv')
df = df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'])

In [3]:
#average rating per user
user_avg_rating = df.groupby('userId')['rating'].mean()

#average rating for user per genre
genre_cols = ['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 
              'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 
              'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']

# Step 1: Multiply each genre column by rating
weighted = df[genre_cols].multiply(df['rating'], axis=0)

# Step 2: Sum ratings per user per genre
genre_sum = weighted.groupby(df['userId']).sum()

# Step 3: Count how many ratings per genre per user
genre_count = df[genre_cols].groupby(df['userId']).sum()

# Step 4: Compute averages
genre_avg = genre_sum / genre_count

user_features = genre_avg.reset_index()

user_features = user_features.rename(
    columns={col: f"user_avg_{col}" for col in genre_cols}
)

In [4]:
# Movie average rating
movie_sum = df.groupby('movieId')['rating'].transform('sum')
movie_count = df.groupby('movieId')['rating'].transform('count')

df['movie_avg_loo'] = (movie_sum - df['rating']) / (movie_count - 1)

# Handle edge case (movies with only 1 rating)
global_avg = df['rating'].mean()
df['movie_avg_loo'] = df['movie_avg_loo'].fillna(global_avg)

In [5]:
#Merge dataframes
# Columns corresponding to the user features
user_cols = [col for col in user_features.columns if col.startswith('user_avg_')]

df = df.merge(user_features, on='userId', how='left')

# Compute global average of ratings
global_avg = df['rating'].mean()

# Add missing indicators
for col in user_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

# Fill NaNs with global average
df[user_cols] = df[user_cols].fillna(global_avg)

# Fill NaNs in movie_avg_loo
df['movie_avg_loo'] = df['movie_avg_loo'].fillna(global_avg)

In [52]:
df.columns

Index(['userId', 'movieId', 'title', 'rating', 'rating_timestamp', 'tag',
       'tag_timestamp', '(no genres listed)', 'Action', 'Adventure',
       'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama',
       'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery',
       'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western', 'two_tower_score',
       'movie_avg_loo', 'user_avg_(no genres listed)', 'user_avg_Action',
       'user_avg_Adventure', 'user_avg_Animation', 'user_avg_Children',
       'user_avg_Comedy', 'user_avg_Crime', 'user_avg_Documentary',
       'user_avg_Drama', 'user_avg_Fantasy', 'user_avg_Film-Noir',
       'user_avg_Horror', 'user_avg_IMAX', 'user_avg_Musical',
       'user_avg_Mystery', 'user_avg_Romance', 'user_avg_Sci-Fi',
       'user_avg_Thriller', 'user_avg_War', 'user_avg_Western',
       'user_avg_(no genres listed)_missing', 'user_avg_Action_missing',
       'user_avg_Adventure_missing', 'user_avg_Animation_missing',
       'user_avg_Chi

In [114]:
# Make sure genre booleans are numeric
global_avg = df['rating'].mean()
for g in genre_cols:
    df[f'user_avg_{g}'] = df[f'user_avg_{g}'].fillna(global_avg)
    df[g] = df[g].astype(int)
    df[f'interact_{g}'] = df[f'user_avg_{g}'] * df[g]

In [125]:
k = 5  # tunable constant
df['movie_avg_weighted'] = df['movie_avg_loo'] * (movie_count / (movie_count + k))

interaction_cols = [f'interact_{g}' for g in genre_cols]
feature_cols = ['two_tower_score', 'movie_avg_loo'] + interaction_cols

In [126]:
# make neural network dataset
X = df[feature_cols]
y = df['rating'].values

scaler = StandardScaler()
X = scaler.fit_transform(X)

user_ids = df['userId'].values
movie_ids = df['movieId'].values

n_users = df['user_idx'].nunique()
n_movies = df['movie_idx'].nunique()

# Split into train/test
X_train, X_test, y_train, y_test, user_train, user_test, movie_train, movie_test = train_test_split(
    X, y, user_ids, movie_ids, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).view(-1,1)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32).view(-1,1)

user_train_tensor = torch.tensor(user_train, dtype=torch.long)
user_test_tensor  = torch.tensor(user_test, dtype=torch.long)
movie_train_tensor = torch.tensor(movie_train, dtype=torch.long)
movie_test_tensor  = torch.tensor(movie_test, dtype=torch.long)

In [127]:
class MovieRecommenderEmb(nn.Module):
    def __init__(self, num_users, num_movies, embed_dim, input_dim):
        super().__init__()
        
        # Embeddings
        self.user_embed = nn.Embedding(num_users, embed_dim)
        self.movie_embed = nn.Embedding(num_movies, embed_dim)
        
        # Fully connected layers
        self.fc = nn.Sequential(
            nn.Linear(input_dim + 2*embed_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, numeric_x, user_x, movie_x):
        user_vec = self.user_embed(user_x)
        movie_vec = self.movie_embed(movie_x)
        x = torch.cat([numeric_x, user_vec, movie_vec], dim=1)
        return self.fc(x)

In [128]:
input_dim = X_train_tensor.shape[1]
num_users = df['userId'].max() + 1
num_movies = df['movieId'].max() + 1
embed_dim = 16

model = MovieRecommenderEmb(num_users, num_movies, embed_dim, input_dim)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-5
)

best_test_loss = float('inf')
patience = 10
counter = 0
epochs = 200

for epoch in range(epochs):
    model.train()
    preds = model(X_train_tensor, user_train_tensor, movie_train_tensor)
    loss = criterion(preds, y_train_tensor)
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    model.eval()
    with torch.no_grad():
        test_preds = model(X_test_tensor, user_test_tensor, movie_test_tensor)
        test_loss = criterion(test_preds, y_test_tensor)
    
    scheduler.step(test_loss)
    
    if test_loss < best_test_loss - 1e-4:
        best_test_loss = test_loss
        best_model_state = model.state_dict()
        counter = 0
    else:
        counter += 1
    
    if (epoch+1) % 10 == 0:
        print(f"Epoch {epoch+1}, Train: {loss.item():.4f}, Test: {test_loss.item():.4f}")
    
    if counter >= patience:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

# Restore best model
model.load_state_dict(best_model_state)

Epoch 10, Train: 12.8210, Test: 12.6587
Epoch 20, Train: 10.6367, Test: 10.4095
Epoch 30, Train: 7.6766, Test: 7.3933
Epoch 40, Train: 4.3259, Test: 4.0709
Epoch 50, Train: 1.8246, Test: 1.7392
Epoch 60, Train: 1.1653, Test: 1.2109
Epoch 70, Train: 0.9974, Test: 1.0516
Epoch 80, Train: 0.8287, Test: 0.9268
Epoch 90, Train: 0.7997, Test: 0.9160
Epoch 100, Train: 0.7716, Test: 0.8972
Epoch 110, Train: 0.7515, Test: 0.8858
Epoch 120, Train: 0.7366, Test: 0.8781
Epoch 130, Train: 0.7234, Test: 0.8714
Epoch 140, Train: 0.7122, Test: 0.8657
Epoch 150, Train: 0.7021, Test: 0.8605
Epoch 160, Train: 0.6931, Test: 0.8558
Epoch 170, Train: 0.6848, Test: 0.8515
Epoch 180, Train: 0.6773, Test: 0.8474
Epoch 190, Train: 0.6703, Test: 0.8434
Epoch 200, Train: 0.6638, Test: 0.8397


<All keys matched successfully>

In [150]:
def predict_top_movies(user_id, df, model, scaler, genre_cols, top_n=5):
    """
    Predict top N movies for a given user, returning movieId, title, and predicted rating.
    """
    # Check user exists
    if user_id not in df['userId'].values:
        raise ValueError(f"User {user_id} not found in dataset.")
    
    # Get user embedding index
    user_idx_val = df.loc[df['userId'] == user_id, 'user_idx'].iloc[0]
    if user_idx_val >= model.user_embed.num_embeddings:
        raise ValueError(f"User index {user_idx_val} is out of range for embeddings.")
    
    # Build feature columns
    interaction_cols = [f'interact_{g}' for g in genre_cols]
    feature_cols = ['two_tower_score', 'movie_avg_loo'] + interaction_cols
    
    # Candidate movies = all movies not already rated by this user
    rated_movies = df[df['userId'] == user_id]['movieId'].tolist()
    movie_candidates = df[['movieId', 'title', 'movie_idx'] + feature_cols].drop_duplicates('movieId')
    movie_candidates = movie_candidates[~movie_candidates['movieId'].isin(rated_movies)]
    
    # Ensure movie indices are valid
    movie_candidates = movie_candidates[movie_candidates['movie_idx'] < model.movie_embed.num_embeddings]
    
    # Extract features and scale
    X_candidates = movie_candidates[feature_cols].values
    X_candidates_scaled = scaler.transform(X_candidates)
    X_tensor = torch.tensor(X_candidates_scaled, dtype=torch.float32)
    
    # User/movie tensors
    movie_idx_tensor = torch.tensor(movie_candidates['movie_idx'].values, dtype=torch.long)
    user_idx_tensor = torch.tensor([user_idx_val]*len(movie_candidates), dtype=torch.long)
    
    # Predict
    model.eval()
    with torch.no_grad():
        preds = torch.clamp(model(X_tensor, user_idx_tensor, movie_idx_tensor), 1, 5).numpy().flatten()
    
    # Attach predictions and return top N
    movie_candidates['pred_rating'] = preds
    top_movies = movie_candidates.sort_values('pred_rating', ascending=False).head(top_n)
    return top_movies[['movieId', 'title', 'pred_rating']]

In [159]:
top5 = predict_top_movies(
    user_id=3, 
    df=df, 
    model=model, 
    scaler=scaler, 
    genre_cols=genre_cols, 
    top_n=15
)
print(top5)

       movieId                                        title  pred_rating
35         596                             Pinocchio (1940)          5.0
39         673                             Space Jam (1996)          5.0
38         661             James and the Giant Peach (1996)          5.0
44         919                     Wizard of Oz, The (1939)          5.0
3100      3159                         Fantasia 2000 (1999)          5.0
3231      6345                        Chorus Line, A (1985)          5.0
3142      4016             Emperor's New Groove, The (2000)          5.0
10514     3201                      Five Easy Pieces (1970)          5.0
9708     77800                            Four Lions (2010)          5.0
7870     54995                         Planet Terror (2007)          5.0
9447    174053         Black Mirror: White Christmas (2014)          5.0
51        1024                 Three Caballeros, The (1945)          5.0
50        1023  Winnie the Pooh and the Blustery Da

c:\Users\atwoo\BYUClasses\Stat486\STAT_486_final_proj\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [164]:
def rescale_predictions_user(preds):
    """
    Rescale predictions to 0-5 per user.
    """
    min_pred = preds.min()
    max_pred = preds.max()
    
    if max_pred - min_pred < 1e-6:
        # all predictions almost identical → assign middle value
        return np.full_like(preds, 2.5)
    
    return 5 * (preds - min_pred) / (max_pred - min_pred)

In [165]:
def predict_top_movies_scaled(user_id, df, model, scaler, genre_cols, top_n=5):
    # Same setup as before
    user_idx_val = df.loc[df['userId'] == user_id, 'user_idx'].iloc[0]
    interaction_cols = [f'interact_{g}' for g in genre_cols]
    feature_cols = ['two_tower_score', 'movie_avg_loo'] + interaction_cols
    
    rated_movies = df[df['userId'] == user_id]['movieId'].tolist()
    movie_candidates = df[['movieId', 'title', 'movie_idx'] + feature_cols].drop_duplicates('movieId')
    movie_candidates = movie_candidates[~movie_candidates['movieId'].isin(rated_movies)]
    movie_candidates = movie_candidates[movie_candidates['movie_idx'] < model.movie_embed.num_embeddings]
    
    X_candidates = movie_candidates[feature_cols].values
    X_candidates_scaled = scaler.transform(X_candidates)
    
    X_tensor = torch.tensor(X_candidates_scaled, dtype=torch.float32)
    movie_idx_tensor = torch.tensor(movie_candidates['movie_idx'].values, dtype=torch.long)
    user_idx_tensor = torch.tensor([user_idx_val]*len(movie_candidates), dtype=torch.long)
    
    # Predict raw ratings
    model.eval()
    with torch.no_grad():
        raw_preds = model(X_tensor, user_idx_tensor, movie_idx_tensor).numpy().flatten()
    
    # Rescale to 0-5 instead of hard clamping
    # After raw predictions
    scaled_preds = rescale_predictions_user(raw_preds)
    movie_candidates['pred_rating'] = scaled_preds
    top_movies = movie_candidates.sort_values('pred_rating', ascending=False).head(top_n)
    
    return top_movies[['movieId', 'title', 'pred_rating']]

In [ ]:
top5 = predict_top_movies_scaled(
    user_id=2, 
    df=df, 
    model=model, 
    scaler=scaler, 
    genre_cols=genre_cols, 
    top_n=5
)
print(top5)

       movieId                                              title  pred_rating
17572     7096                            Rivers and Tides (2001)     5.000000
14263   124404                Snowflake, the White Gorilla (2011)     4.900941
24667     4406           Man Who Shot Liberty Valance, The (1962)     4.579940
9447    174053               Black Mirror: White Christmas (2014)     4.560266
41322     4334                                       Yi Yi (2000)     4.398891
41394     6254                            Awful Truth, The (1937)     4.398824
1710      1209  Once Upon a Time in the West (C'era una volta ...     4.369081
50        1023        Winnie the Pooh and the Blustery Day (1968)     4.364315
15756     5380            Importance of Being Earnest, The (2002)     4.355250
19708    50842  Boss of It All, The (Direktøren for det hele) ...     4.350508
72582    82857                                  Sweetgrass (2009)     4.300258
14248   118834                  National Lampoon's B

c:\Users\atwoo\BYUClasses\Stat486\STAT_486_final_proj\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


Next thing to do would be to use candidate filtering where I filter to movies similar to teh user's watched genres and to add a tiny noise term before scaling.